### Pipeline de base pour le clusturing des clients a tester sur les données optimiser par le rfm



In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
print("Dependances de bases pret..")

Dependances de bases pret..


In [3]:
df= pd.read_csv("C:\\Users\\Ce PC\\client-scope-rfm-project\\data.csv", sep=",")
print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df.head())

Données chargées : 794498 lignes, 8 colonnes
  Invoice  StockCode                          Description  Quantity  \
0  489434    85048.0  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323.0                   PINK CHERRY LIGHTS        12   
2  489434    79323.0                  WHITE CHERRY LIGHTS        12   
3  489434    22041.0         RECORD FRAME 7" SINGLE SIZE         48   
4  489434    21232.0       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  CustomerID         Country  
0  2009-12-01 07:45:00   6.95       13085  United Kingdom  
1  2009-12-01 07:45:00   6.75       13085  United Kingdom  
2  2009-12-01 07:45:00   6.75       13085  United Kingdom  
3  2009-12-01 07:45:00   2.10       13085  United Kingdom  
4  2009-12-01 07:45:00   1.25       13085  United Kingdom  


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794498 entries, 0 to 794497
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      794498 non-null  object 
 1   StockCode    794498 non-null  float64
 2   Description  794498 non-null  object 
 3   Quantity     794498 non-null  int64  
 4   InvoiceDate  794498 non-null  object 
 5   Price        794498 non-null  float64
 6   CustomerID   794498 non-null  int64  
 7   Country      794498 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 48.5+ MB


### construction d'un pipeline de base pour le test clustering sur les données nétoyée


### Preparation des données

In [5]:

# 1. Nettoyage des données
#    - Les vraies données de vente contiennent souvent des retours
#      (Quantity négative) et des annulations (Invoice commençant par 'C')
#      qu'il faut retirer avant de calculer le RFM.

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
 
# Retirer les annulations (Invoice commençant par 'C') si présentes
df = df[~df["Invoice"].astype(str).str.startswith("C")]
 
# Retirer les quantités ou prix négatifs/nuls (retours, erreurs)
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
 
# Retirer les lignes sans CustomerID (impossible de rattacher à un client)
df = df.dropna(subset=["CustomerID"])
 
# Calcul du montant par ligne de commande
df["MontantLigne"] = df["Quantity"] * df["Price"]
 
print(f"\nAprès nettoyage : {df.shape[0]} lignes, {df['CustomerID'].nunique()} clients uniques")
 


Après nettoyage : 776840 lignes, 5853 clients uniques


In [6]:

# 2. Calcul du RFM par client

date_reference = df["InvoiceDate"].max() + pd.Timedelta(days=1)  # jour suivant la dernière transaction
 
rfm = df.groupby("CustomerID").agg(
    recence_jours=("InvoiceDate", lambda x: (date_reference - x.max()).days),
    frequence=("Invoice", "nunique"),          # nombre de commandes distinctes
    montant_total=("MontantLigne", "sum")
).reset_index()
 
print("\nAperçu du RFM calculé :")
print(rfm.head())
print(rfm[["recence_jours", "frequence", "montant_total"]].describe())
 


Aperçu du RFM calculé :
   CustomerID  recence_jours  frequence  montant_total
0       12346            326         12       77556.46
1       12347              2          8        4921.53
2       12348             75          5        1658.40
3       12349             19          3        3678.69
4       12350            310          1         294.40
       recence_jours    frequence  montant_total
count    5853.000000  5853.000000    5853.000000
mean      200.249103     6.255425    2918.517986
std       208.528333    12.762146   14335.507948
min         1.000000     1.000000       2.950000
25%        25.000000     1.000000     340.850000
50%        95.000000     3.000000     856.030000
75%       379.000000     7.000000    2240.900000
max       739.000000   375.000000  580987.040000


In [7]:
# 3. Normalisation
#    On applique aussi un log sur montant et fréquence, car ces
#    variables sont souvent très asymétriques (quelques gros clients
#    achètent énormément) -> le log réduit l'effet des valeurs extrêmes.

rfm_log = rfm.copy()
rfm_log["frequence"] = np.log1p(rfm_log["frequence"])
rfm_log["montant_total"] = np.log1p(rfm_log["montant_total"])
 
features = rfm_log[["recence_jours", "frequence", "montant_total"]]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
 

In [8]:

# 4. Méthode du coude pour choisir k
inertias = []
k_range = range(1, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(features_scaled)
    inertias.append(km.inertia_)
 
plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker="o")
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertie")
plt.title("Méthode du coude")
plt.tight_layout()
plt.savefig("methode_coude.png")
plt.close()
 

Exception in thread Thread-3 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Ce PC\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\Ce PC\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\Ce PC\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Ce PC\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\Ce PC\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 

In [9]:
# 5. Clustering final
#    -> Ajuste k_final selon le coude obtenu (souvent 4 à 5 pour du RFM)
k_final = 4
kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
rfm["cluster"] = kmeans.fit_predict(features_scaled)
 

In [10]:

# 6. Interprétation des clusters (sur les valeurs réelles, pas le log)
resume = rfm.groupby("cluster")[["recence_jours", "frequence", "montant_total"]].mean().round(1)
resume["nb_clients"] = rfm.groupby("cluster").size()
resume["% clients"] = (resume["nb_clients"] / len(rfm) * 100).round(1)
 
med_r, med_f, med_m = resume["recence_jours"].median(), resume["frequence"].median(), resume["montant_total"].median()
 
def etiqueter(row):
    if row["recence_jours"] <= med_r and row["montant_total"] >= med_m:
        return "Clients VIP / fidèles"
    elif row["recence_jours"] > med_r and row["frequence"] <= med_f:
        return "Clients perdus / inactifs"
    elif row["frequence"] <= med_f:
        return "Nouveaux clients prometteurs"
    else:
        return "Clients réguliers"
 
resume["profil"] = resume.apply(etiqueter, axis=1)
print("\nProfil moyen et interprétation par cluster :")
print(resume)
 


Profil moyen et interprétation par cluster :
         recence_jours  frequence  montant_total  nb_clients  % clients  \
cluster                                                                   
0                 98.3        5.5         1871.3        1875       32.0   
1                490.8        1.7          517.0        1634       27.9   
2                104.1        1.7          421.3        1397       23.9   
3                 42.5       22.2        12819.5         947       16.2   

                            profil  
cluster                             
0            Clients VIP / fidèles  
1        Clients perdus / inactifs  
2        Clients perdus / inactifs  
3            Clients VIP / fidèles  


In [11]:
# 7. Visualisation

plt.figure(figsize=(7, 5))
scatter = plt.scatter(rfm["frequence"], rfm["montant_total"], c=rfm["cluster"], cmap="viridis", alpha=0.6)
plt.xlabel("Fréquence d'achat (nb commandes)")
plt.ylabel("Montant total dépensé (£)")
plt.title("Segmentation clients (K-means sur RFM)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("clusters_visualisation.png")
plt.close()
 

In [12]:
# 8. Export
rfm_final = rfm.merge(resume[["profil"]], on="cluster")
rfm_final.to_csv("clients_rfm_clusters.csv", index=False)
print("\nFichiers générés : methode_coude.png, clusters_visualisation.png, clients_rfm_clusters.csv")
 


Fichiers générés : methode_coude.png, clusters_visualisation.png, clients_rfm_clusters.csv
